# Mann-Whitney Test for Term Distinctiveness

This is a sample implementation of the Mann-Whitney test (AKA Wilcoxon Rank-Sum test) for term distinctiveness in a corpus of documents. The test is used to determine if the distribution of term frequencies in two different sets of documents is significantly different. It makes a useful alternative to the z-test.

### References

- Jefrey Lijffijt, Terttu Nevalainen, Tanja Säily, et al,
“Significance testing of word frequencies in corpora”,
_Digital Scholarship in the Humanities_ 31, no. 2 (2016):
375-397.
- Andrew Piper and Eva Portelace, [“How Cultural
Capital Works: Prizewinning Novels, Bestsellers, and
the Time of Reading”](http://post45.org/2016/05/how-cultural-capital-works-prizewinning-novels-bestsellers-and-the-time-of-reading/), _Post45_ (May 10, 2016).

In the next cell, we will define a small corpus of documents and divide them into two groups for comparison.

In [1]:
# Import necessary libraries
import pandas as pd
import spacy
from lexos.dtm import DTM

# Load the small English model from spaCy
nlp = spacy.load("en_core_web_sm")

# Sample text documents for testing
texts = [
    "This is a sample text for testing.", # odd
    "Here is another example of a text to analyze.", # even
    "This text is different from the others.", # odd
    "Yet another sample text for comparison.", # even
    "This text is similar to the first one.", # odd
    "A completely different text for the analysis.", # even
]

# Process the sample texts with spaCy to create documents
docs = [nlp(text) for text in texts]
labels = [f"Doc{i + 1}" for i in range(len(docs))]
even_docs = ["Doc2", "Doc4", "Doc6"]
odd_docs = ["Doc1", "Doc3", "Doc5"]

# Create a Document-Term Matrix (DTM) using the sample documents
# Limit to terms occurring in at least 2 documents
dtm = DTM()
dtm(docs=docs, labels=labels, min_df=2)

# Convert the DTM to a DataFrame and separate even and odd documents
df = dtm.to_df(transpose=True)
even_df = df[df.index.isin(even_docs)]
odd_df = df[df.index.isin(odd_docs)]


x_sorted = even_df.T.sort_index(ascending=True)
x_sorted["Mean"] = x_sorted.mean(axis=1)
x_sorted

,Doc2,Doc4,Doc6,Mean
.,1,1,1,1
This,0,0,0,0
a,1,0,0,0
another,1,1,0,0
different,0,0,1,0
for,0,1,1,0
is,1,0,0,0
sample,0,1,0,0
text,1,1,1,1
the,0,0,1,0


In the next cell, we define a function to calculate the Mann Whitney U statistic and its p-value. The function takes two dataframes as input. The first is the data grouping for which the the results of the test will be reported. The second is the the data grouping to which the first will be compared. The function returns a ranking statistic and a p-value for each term. The highest ranked terms are the ones that are most distinctive to the first data grouping. An additional function provides the option to add some statistics showing the relative frequency of the terms between each grouping.

In [2]:
# Import necessary libraries
import pandas as pd
from scipy.stats import mannwhitneyu
from wasabi import msg

from lexos.exceptions import LexosException


def get_freq_stats(x: pd.DataFrame, y: pd.DataFrame) -> pd.DataFrame:
    """Calculates the mean frequencies for control and comparison DataFrames.

    Also computes the difference in frequency from control to comparison.

    Args:
      x: DataFrame containing frequencies for control documents.
      y: DataFrame containing frequencies for compare documents.

    Returns:
      A DataFrame with means and differences in frequency.
    """
    x_sorted = x.T.sort_index(ascending=True)
    x_sorted["Mean"] = x_sorted.mean(axis=1)
    x_mean = x_sorted["Mean"].tolist()

    y_sorted = y.T.sort_index(ascending=True)
    y_sorted["Mean"] = y_sorted.mean(axis=1)
    y_mean = y_sorted["Mean"].tolist()

    difference = [v1 - v2 for v1, v2 in zip(x_mean, y_mean)]
    difference = [f"{d*100:.2f}%" for d in difference]
    df = pd.DataFrame({"ave_freq": x_mean, "difference": difference})
    df.index = x_sorted.index
    return df


def calculate_mannwhitney_u(
    x: pd.DataFrame, y: pd.DataFrame, add_freq: bool = True
) -> pd.DataFrame:
    """Calculates the Mann-Whitney U test for corresponding columns in two DataFrames.

    Args:
      x: DataFrame containing data for which term salience will be reported.
        Assumes numeric data in columns where the test is applicable.
      y: DataFrame containing data to which the data in x will be compared.
        Assumes numeric data and the same column names as even_df.
      add_freq: If True, adds average frequency and increase in frequency.

    Returns:
      A Pandas DataFrame with columns 'term', 'statistic', and 'p_value'
      reporting the Mann-Whitney U test results for each common term.
      Returns an empty DataFrame if no common terms with numeric data
      are found or if input DataFrames are unsuitable.
    """
    results = []
    if not isinstance(x, pd.DataFrame) or not isinstance(y, pd.DataFrame):
        raise LexosException("Error: Inputs must be Pandas DataFrames.")
        return pd.DataFrame(columns=["term", "statistic", "p_value"])

    if x.empty or y.empty:
        msg.warn("Warning: One or both input DataFrames are empty.")
        return pd.DataFrame(columns=["term", "statistic", "p_value"])

    # Iterate through columns of the first DataFrame
    # Assumes odd_df has corresponding columns
    for col in x.columns:
        if col not in y.columns:
            msg.warn(f"Warning: Column '{col}' not found in y. Skipping.")
            continue

        # Extract data for the current column
        x_data = x[col].dropna()  # Drop NaNs as mannwhitneyu can't handle them
        y_data = y[col].dropna()

        # Ensure data is numeric and there's enough data to perform the test
        if not pd.api.types.is_numeric_dtype(
            x_data
        ) or not pd.api.types.is_numeric_dtype(y_data):
            msg.warn(
                f"Warning: Column '{col}' is not numeric in one or both DataFrames. Skipping."
            )
            continue

        if len(x_data) == 0 or len(y_data) == 0:
            msg.warn(
                f"Warning: Column '{col}' has no non-NaN data in one or both groups after dropping NaNs. Skipping."
            )
            continue

        try:
            # Perform the Mann-Whitney U test
            # Use alternative="two-sided" for a standard two-tailed test
            statistic, p_value = mannwhitneyu(x_data, y_data, alternative="two-sided")
            results.append({"term": col, "statistic": statistic, "p_value": p_value})
        except ValueError as e:
            # This can happen if all values are identical in one or both samples,
            # or other edge cases.
            LexosException(
                f"Could not calculate Mann-Whitney U for column '{col}': {e}"
            )
            results.append(
                {"term": col, "statistic": float("nan"), "p_value": float("nan")}
            )

    if not results:
        LexosException("No suitable columns found to perform the Mann-Whitney U test.")
        return pd.DataFrame(columns=["term", "statistic", "p_value"])

    result = pd.DataFrame(results)

    if add_freq:
        # Make sure that the Mann-Whitney U results are sorted by term
        result = result.sort_values(by="term", ascending=True)

        # Get the control mean and difference
        freq_stats = get_freq_stats(x, y)

        # Add the average frequency and difference to the result DataFrame
        result["ave_freq"] = freq_stats["ave_freq"].tolist()
        result["difference"] = freq_stats["difference"].tolist()

    # Sort by the statistic
    return result.sort_values(by="statistic", ascending=False)

Now we call the function with our two dataframes and print the results. The output will show the terms ranked by their distinctiveness, along with their U statistic and p-value.

The p-value is the probability that a test statistic is extreme or more extreme than the one observed, assuming that the two samples come from the same distribution. A small p-value (typically less than 0.05) indicates that the observed difference between the two samples is statistically significant, and we conclude that the two samples do not come from the same distribution.

By default, the table displays the average frequency of terms in the control group along with the increase in frequency in the comparison group. This provides us with another view of how important the word is to the sample and its relative over- or under-usage in comparison to the other sample.


In [3]:
result = calculate_mannwhitney_u(even_df, odd_df)
print("**Results of Mann-Whitney U Test**")
print("Average frequency is for the control group and the difference is between control and comparison.")
result

**Results of Mann-Whitney U Test**
Average frequency is for the control group and the difference is between control and comparison.


,term,statistic,p_value,ave_freq,difference
0,another,7.5,0.187632,0,0.00%
7,for,6.0,0.619257,0,0.00%
4,.,4.5,1.000000,1,0.00%
6,a,4.5,1.000000,0,0.00%
1,different,4.5,1.000000,0,0.00%
9,sample,4.5,1.000000,0,0.00%
10,text,4.5,1.000000,1,0.00%
3,to,4.5,1.000000,0,0.00%
2,the,3.0,0.619257,0,0.00%
8,is,1.5,0.187632,0,-100.00%
